[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Zgraph/blob/main/zgraph/examples/stoichiometric.ipynb)

In [2]:
# Install Zgraph if running in Google Colab
try:
    from zgraph import *
except ImportError:
    !pip install -q "git+https://github.com/themintlab/Zgraph.git#subdirectory=zgraph"
    from zraph import *

In [3]:
import jax
import jax.numpy as jnp
import numpy as np
from zgraph import *
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
T, mu1, mu2 = SignalNodes(0, 1, 2)

In [5]:
R = 8.314
RT = FactorNode([[R]], [T])
mu1A = FactorNode([1, -1], [RT, mu1])
mu1B = FactorNode([-1], [mu1])
mu2B = FactorNode([1, -1], [RT, mu2])

In [6]:
phaseA = FactorNode(jnp.eye(1), [mu1A], beta=RT)
phaseB = FactorNode(jnp.eye(2), [mu1B, mu2B], beta=RT)
system = FactorNode(jnp.eye(2), [phaseA, phaseB], beta=0)

# Grand potential plot

In [7]:
T_val = jnp.array(293.15)
mu1 = jnp.linspace(-10 * R * 300, 10 * R * 300, num=500)
mu2 = jnp.zeros_like(mu1)
T_flat = T_val * jnp.ones_like(mu1)
input_tensor = jnp.stack([T_flat, mu1, mu2], axis=-1)

In [8]:
primals = [phaseA, phaseB, system]
phaseAb, phaseBb, systemb = graph_to_function(primals, compile=True)

In [9]:
gA_vals = phaseAb(input_tensor).squeeze()
gB_vals = phaseBb(input_tensor).squeeze()
gsys_vals = systemb(input_tensor).squeeze()
x_mu = input_tensor[..., 1].squeeze()

# Free energy plot

In [10]:
fcns = legendre_transform(primals, [1, 2])
fA, fB, fsys = graph_to_function(fcns, compile=True)

shifted_coords = gauge_fix(systemb, input_tensor, [1, 2])
fAd, muAd = fA(shifted_coords)

fBd, muBd = fB(shifted_coords)

fsysd, musysd = fsys(shifted_coords)

In [11]:
xA_frac = -muAd[:, 1].squeeze()
yA_free = -fAd.squeeze()

xB_frac = -muBd[:, 1].squeeze()
yB_free = -fBd.squeeze()

xsys_frac = -musysd[:, 1].squeeze()
ysys_free = -fsysd.squeeze()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=("Grand potential plot", "Free energy plot")
)

colors = {
    "phase A": "#1f77b4",
    "phase B": "#ff7f0e",
    "Equilibrium": "#2ca02c"
}

fig.add_trace(
    go.Scatter(
        x=x_mu, y=gA_vals,
        name="phase A",
        legendgroup="phase A",
        showlegend=True,
        line=dict(width=3, color=colors["phase A"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gB_vals,
        name="phase B",
        legendgroup="phase B",
        showlegend=True,
        line=dict(width=3, color=colors["phase B"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gsys_vals,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=True,
        line=dict(width=3, color=colors["Equilibrium"], dash='dash')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=xA_frac, y=yA_free,
        name="phase A",
        legendgroup="phase A",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase A"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xB_frac, y=yB_free,
        name="phase B",
        legendgroup="phase B",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase B"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xsys_frac, y=ysys_free,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["Equilibrium"], dash='dash')
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Chemical potential difference, Î”Î¼", row=1, col=1)
fig.update_yaxes(title_text="Grand potential, Î©", row=1, col=1)

fig.update_xaxes(title_text="Mole fraction", row=2, col=1)
fig.update_yaxes(title_text="Free energy", row=2, col=1)

fig.update_layout(
    height=750,
    width=800,
    template="plotly_white"
)

fig.show()